In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
import torch.nn as nn
import torch.optim as optim

from helper_modules import phase1_preprocess_data, phase2_preprocess_data

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)


### Read in the (test/submission) data

In [ ]:
df_test = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_test.csv'))
display(df_test.head())

### Stage 1 Predictions -- Classifying Demand_Response_Flag

In [ ]:
# perform some preprocessing for phase 1
df = phase1_preprocess_data(df_test)

In [ ]:
# Load the trained model from file
from net_architecture import Net    # Define the neural network architecture: Just need to load architecture defined in net_architecture.py
input_dim = df.shape[1]             # Number of features
num_classes = 3                     # Number of classes in target variable
model_loaded = Net(input_dim, num_classes)
model_loaded.load_state_dict(torch.load('./models/phase1_nn_model.pth'))

# load the scaler, which should be ColumnTransformer type
# scaler is a combination of PowerTransformer (Yeo-Johnson transformation) & StandardScaler
scaler = joblib.load('./models/phase1_scaler.pkl')
print(scaler)

In [ ]:
# Convert DataFrame to torch tensor
X = scaler.transform(df)
X = torch.tensor(X, dtype=torch.float32)

# Set model to evaluation mode
model_loaded.eval()
with torch.no_grad():
    outputs = model_loaded(X)
    predictions = torch.argmax(outputs, dim=1)

# convert 2 in predictions to -1
predictions = np.where(predictions == 2, -1, predictions)

# Calculate the distribution of predictions classes
unique, counts = np.unique(predictions, return_counts=True)
distribution_pred = dict(zip(unique, counts))
print(distribution_pred)

In [ ]:
# Prepare data for Stage 2 predictions
df_stage2 = df_test.copy(deep=True)
df_stage2['Demand_Response_Flag'] = predictions

### Stage 2 Predictions - Demand_Response_Capacity_kW

In [ ]:
# perform some preprocessing for phase 2
df = phase2_preprocess_data(df_stage2)
df.head()

In [ ]:
# Load models
clf_nz = joblib.load('./models/phase2_xgb_clf_nz_model.pkl')
reg_nz = joblib.load('./models/phase2_xgb_reg_nz_model.pkl')

# Prepare test features (X_test)
X_test = df.values

# Predict on real test set
p_nz = clf_nz.predict_proba(X_test)[:, 1]
mu_nz = reg_nz.predict(X_test)
y_pred_xgb = p_nz * mu_nz


In [ ]:
df_submission = df_test[['Site', 'Timestamp_Local']].copy(deep=True)
df_submission['Demand_Response_Flag'] = predictions
df_submission['Demand_Response_Capacity_kW'] = y_pred_xgb

# manually enforce 0 for Demand_Response_Capacity_kW if Demand_Response_Flag = 0
df_submission.loc[df_submission['Demand_Response_Flag'] == 0, 'Demand_Response_Capacity_kW'] = 0

df_submission.to_csv('./submissions/submission.csv', index=False)
print("Submission file created: submission.csv in ./submissions folder")

In [ ]:
display(df_submission.head())
print('submission file shape:', df_submission.shape)